In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from statsmodels.stats.contingency_tables import mcnemar

sns.set_theme(style="darkgrid")

In [3]:
data_path = r'C:\Users\adity\Desktop\Kepler-Exoplanet Project'
df = pd.read_csv(os.path.join(data_path, "data", "kepler_clean.csv"))

In [4]:
TARGET = 'koi_disposition'
df_model = df[df[TARGET].isin(['CONFIRMED', 'FALSE POSITIVE'])].copy()

KEY_FEATURES = [
    'koi_period', 'koi_prad', 'koi_teq', 'koi_insol',
    'koi_model_snr', 'koi_steff', 'koi_slogg',
    'koi_srad', 'koi_score', 'koi_depth',
    'koi_duration', 'koi_impact',
    'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec'
]

X = df_model[KEY_FEATURES]
y_raw = df_model[TARGET]

le = LabelEncoder()
y = le.fit_transform(y_raw)

imputer = SimpleImputer(strategy='median')

df_model['star_id'] = df_model['ra'].round(5).astype(str) + '_' + df_model['dec'].round(5).astype(str)
groups = df_model['star_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_imp, y_train)

FEATURE_TIERS = {
    'Full': KEY_FEATURES,
    'Raw physical': [f for f in KEY_FEATURES if f not in
                      ['koi_score', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec']]
}

print(f"Train size: {len(X_train):,} | Test size: {len(X_test):,}")

Train size: 4,941 | Test size: 1,262


LOGISTIC REGRESSION

In [5]:
print("LOGISTIC REGRESSION - Baseline Across Feature Tiers")
print("=" * 70)

logreg_results = {}

for tier_name, features in FEATURE_TIERS.items():
    X_train_tier = pd.DataFrame(X_train_bal, columns=KEY_FEATURES)[features]
    X_test_tier = pd.DataFrame(X_test_imp, columns=KEY_FEATURES)[features]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_tier)
    X_test_scaled = scaler.transform(X_test_tier)

    logreg = LogisticRegression(max_iter=1000, random_state=42)
    logreg.fit(X_train_scaled, y_train_bal)

    y_pred = logreg.predict(X_test_scaled)
    y_prob = logreg.predict_proba(X_test_scaled)[:, 1]

    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    logreg_results[tier_name] = {
        'model': logreg, 'y_pred': y_pred, 'y_prob': y_prob,
        'f1': f1, 'auc': auc, 'features': features
    }

    print(f"\n{tier_name} ({len(features)} features)")
    print(f" F1={f1:.4f} | AUC={auc:.4f}")

    coefs = pd.Series(logreg.coef_[0], index = features).sort_values(key=abs, ascending=False)
    print(" Top coefficients (standardized, sign = direction toward FALSE POSITIVE):")
    print(coefs.head(6).to_string())

LOGISTIC REGRESSION - Baseline Across Feature Tiers

Full (16 features)
 F1=0.9961 | AUC=0.9987
 Top coefficients (standardized, sign = direction toward FALSE POSITIVE):
koi_fpflag_co    3.269995
koi_score       -2.698610
koi_fpflag_ss    2.686339
koi_fpflag_ec    2.320390
koi_depth        2.239404
koi_fpflag_nt    2.160701

Raw physical (11 features)
 F1=0.8622 | AUC=0.9280
 Top coefficients (standardized, sign = direction toward FALSE POSITIVE):
koi_prad      7.332709
koi_depth     6.857060
koi_srad      4.149036
koi_teq       2.586334
koi_slogg     1.543528
koi_impact    1.488289


In [6]:
print("LightGBM - Raw Physical Tier")
print("=" * 70)

raw_features = FEATURE_TIERS['Raw physical']
X_train_raw = pd.DataFrame(X_train_bal, columns=KEY_FEATURES)[raw_features]
X_test_raw = pd.DataFrame(X_test_imp, columns=KEY_FEATURES)[raw_features]

lgbm = LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbose=-1
)
lgbm.fit(X_train_raw, y_train_bal)

y_pred_lgbm = lgbm.predict(X_test_raw)
y_prob_lgbm = lgbm.predict_proba(X_test_raw)[:,1]

f1_lgbm = f1_score(y_test, y_pred_lgbm)
auc_lgbm = roc_auc_score(y_test, y_prob_lgbm)

print(f"LightGBM (raw physical) - F1={f1_lgbm:.4f} | AUC={auc_lgbm:.4f}")
print(f"\nFor comparison, from Phase 8 (raw physical tier)")
print(f"  Random Forest      F1=0.9331 | AUC=0.9747")
print(f"  XGBoost            F1=0.9412 | AUC=0.9786")
print(f"  Gradient Boosting  F1=0.9297 | AUC=0.9749")
print(f"  SVM                F1=0.8875 | AUC=0.9535")

LightGBM - Raw Physical Tier
LightGBM (raw physical) - F1=0.9382 | AUC=0.9781

For comparison, from Phase 8 (raw physical tier)
  Random Forest      F1=0.9331 | AUC=0.9747
  XGBoost            F1=0.9412 | AUC=0.9786
  Gradient Boosting  F1=0.9297 | AUC=0.9749
  SVM                F1=0.8875 | AUC=0.9535
